# HealthGuard AI — Phase 5: Fine-tuned DistilBERT vs Classical NLP Baselines

**Module:** 6WCM0029 Final Year Project
**Author:** Muhammad Zubair, University of Hertfordshire

## Why this phase exists
This is the second **pretrained** model in the project (after TabPFN), satisfying
supervisor point (1). It extends HealthGuard AI from structured clinical records
into free-text symptom intake — the modality a real triage system actually receives
from patients.

## The question being asked
Not *"can DistilBERT classify symptoms?"* — it obviously can. The question is:

> **Does a 66M-parameter pretrained transformer beat TF-IDF + Logistic Regression
> on a 1,200-row clinical text dataset, and if so, by how much and at what cost?**

That mirrors the question already answered twice in this project:
- **Tabular:** gradient boosting beat deep learning (918 rows) — Grinsztajn et al. (2022)
- **ECG:** CNN beat BiLSTM (78,798 beats) — morphology, not sequence
- **Text:** this phase

## Models compared
| Model | Type | Params | What it tests |
|---|---|---|---|
| TF-IDF + Logistic Regression | Classical, linear | ~30k | Is lexical overlap sufficient? |
| TF-IDF + Linear SVM | Classical, linear | ~30k | Margin-based alternative |
| Frozen DistilBERT + LR | Pretrained, no fine-tuning | 0 trained | Do the pretrained features alone help? |
| **Fine-tuned DistilBERT** | Pretrained, fine-tuned | 66M | Does adapting the weights pay off? |

The **frozen** condition is the important control. Without it you cannot tell whether
any gain comes from the pretrained representation or from fine-tuning on your data.

## Headline metric: macro F1
24 classes. Consistent with the ECG phase, so the two chapters compare cleanly.

## The learning curve
Section 9 retrains everything on 10%, 25%, 50% and 100% of the training data.
This locates the point where pretrained knowledge starts to pay for itself — a
direct, empirical answer to *"when is a transformer worth it?"* and the most
report-worthy figure in this notebook.

---
## 1. Environment
Runtime → Change runtime type → **T4 GPU** before running anything.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then re-run."

In [ ]:
!pip install -q transformers==4.44.2 2>/dev/null
print("transformers installed")

In [ ]:
import os, json, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, accuracy_score,
                             precision_recall_fscore_support)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = torch.device('cuda')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
print("Setup complete. Device:", DEVICE)

---
## 2. Dataset

**Source:** `niyarrbarman/symptom2disease` on Kaggle — 1,200 natural-language symptom
descriptions written in first person, labelled with one of 24 conditions
(50 examples per class, perfectly balanced).

Example: *"I have been experiencing a skin rash on my arms, legs, and torso for the
past few weeks. It is red, itchy, and covered in dry, scaly patches."* → Fungal infection

**Why this dataset fits HealthGuard AI:** it is the input a patient-facing triage
form actually produces. Your tabular models need a clinician to have already
measured cholesterol and ST slope; this model works from what the patient types.

**Get your Kaggle token:** kaggle.com → avatar → Settings → API → Create New Token.

In [ ]:
from google.colab import files

os.makedirs('/root/.kaggle', exist_ok=True)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Upload your kaggle.json:")
    up = files.upload()
    with open('/root/.kaggle/kaggle.json','wb') as f:
        f.write(list(up.values())[0])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!pip install -q kaggle
!kaggle datasets download -d niyarrbarman/symptom2disease -p /content/data --unzip
!ls -la /content/data

In [ ]:
import glob
csvs = glob.glob('/content/data/*.csv')
print("Found:", csvs)
df = pd.read_csv(csvs[0])
print("\nColumns:", list(df.columns))
print("Shape:", df.shape)

# Column names vary between releases - normalise defensively
cols_lower = {c.lower().strip(): c for c in df.columns}
text_col = next((cols_lower[k] for k in ['text','symptoms','symptom','description']
                 if k in cols_lower), None)
label_col = next((cols_lower[k] for k in ['label','disease','diagnosis','class']
                  if k in cols_lower), None)
assert text_col and label_col, f"Could not identify columns in {list(df.columns)}"

df = df[[text_col, label_col]].rename(columns={text_col:'text', label_col:'label'})
df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"\nUsing text='{text_col}', label='{label_col}'")
print(f"After dropping nulls/duplicates: {len(df):,} rows, {df.label.nunique()} classes")
df.head(3)

In [ ]:
counts = df.label.value_counts()
print("Class balance:")
print(f"  classes: {len(counts)}   min: {counts.min()}   max: {counts.max()}")

df['n_words'] = df.text.str.split().str.len()
print(f"\nText length (words): median {df.n_words.median():.0f}, "
      f"95th pct {df.n_words.quantile(0.95):.0f}, max {df.n_words.max()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
counts.sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title(f'Class distribution ({len(counts)} conditions)')
axes[0].set_xlabel('examples')
axes[0].tick_params(labelsize=8)
axes[1].hist(df.n_words, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(df.n_words.quantile(0.95), color='crimson', ls='--',
                label=f'95th pct = {df.n_words.quantile(0.95):.0f} words')
axes[1].set_title('Symptom description length')
axes[1].set_xlabel('words'); axes[1].legend()
plt.tight_layout(); plt.savefig('/content/fig_nlp_eda.png', bbox_inches='tight'); plt.show()

---
## 3. Splits

Stratified **70 / 15 / 15** train / validation / test. Validation is used only for
epoch selection; the test set is scored once, at the very end.

`MAX_LEN = 128` tokens comfortably covers the 95th percentile of description length,
so truncation affects almost nothing — worth stating in the report rather than
leaving as an unexamined default.

In [ ]:
le = LabelEncoder()
y_all = le.fit_transform(df.label.values)
X_all = df.text.values
N_CLASSES = len(le.classes_)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print(f"train {len(X_tr)}   val {len(X_val)}   test {len(X_te)}   classes {N_CLASSES}")

MAX_LEN = 128
RESULTS = {}

def evaluate(name, y_true, y_pred, train_time, n_params, infer_ms):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=range(N_CLASSES), zero_division=0)
    RESULTS[name] = {
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
        'macro_f1': round(float(f1_score(y_true, y_pred, average='macro')), 4),
        'weighted_f1': round(float(f1_score(y_true, y_pred, average='weighted')), 4),
        'train_time_s': round(train_time, 1),
        'trainable_params': int(n_params),
        'inference_ms_per_sample': round(infer_ms, 3),
        'worst_class_f1': round(float(f1.min()), 4),
        'worst_class': str(le.classes_[int(np.argmin(f1))]),
    }
    r = RESULTS[name]
    print(f"\n--- {name} ---")
    print(f"accuracy {r['accuracy']:.4f}   macro F1 {r['macro_f1']:.4f}  <-- headline")
    print(f"train {r['train_time_s']}s   {r['trainable_params']:,} trainable params")
    print(f"weakest class: {r['worst_class']} (F1 {r['worst_class_f1']:.3f})")
    return RESULTS[name]

---
## 4. Baseline 1 & 2 — TF-IDF + linear classifiers

Word and character n-grams. These are the models a transformer has to beat to
justify 66M parameters and a GPU.

In [ ]:
def run_tfidf(name, clf):
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(sublinear_tf=True, ngram_range=(1,2),
                                  min_df=2, stop_words='english')),
        ('clf', clf),
    ])
    t0 = time.time(); pipe.fit(X_tr, y_tr); t = time.time() - t0
    t1 = time.time(); pred = pipe.predict(X_te)
    infer = 1000*(time.time()-t1)/len(X_te)
    n_feat = len(pipe.named_steps['tfidf'].vocabulary_)
    evaluate(name, y_te, pred, t, n_feat * N_CLASSES, infer)
    return pipe, pred

lr_pipe,  lr_pred  = run_tfidf('TF-IDF + LogReg',
                               LogisticRegression(max_iter=2000, C=10, random_state=SEED))
svm_pipe, svm_pred = run_tfidf('TF-IDF + LinearSVM',
                               LinearSVC(C=1.0, random_state=SEED))

---
## 5. Baseline 3 — Frozen DistilBERT embeddings + Logistic Regression

The control condition. DistilBERT produces sentence embeddings (mean-pooled over
tokens, attention-masked), but **no weights are updated**. A logistic regression is
fitted on those fixed 768-dimensional vectors.

If this matches fine-tuned DistilBERT, fine-tuning added nothing.
If it collapses, the pretrained representation alone is insufficient for clinical text.
Either result is informative — which is why the control belongs in the report.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

@torch.no_grad()
def embed(texts, batch_size=64):
    enc_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    out = []
    for i in range(0, len(texts), batch_size):
        b = list(texts[i:i+batch_size])
        enc = tokenizer(b, truncation=True, padding=True,
                        max_length=MAX_LEN, return_tensors='pt').to(DEVICE)
        h = enc_model(**enc).last_hidden_state            # (B, T, 768)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1)          # masked mean pooling
        out.append(pooled.cpu().numpy())
    del enc_model; torch.cuda.empty_cache()
    return np.vstack(out)

t0 = time.time()
E_tr, E_te = embed(X_tr), embed(X_te)
embed_time = time.time() - t0

t0 = time.time()
frozen_clf = LogisticRegression(max_iter=3000, C=10, random_state=SEED).fit(E_tr, y_tr)
fit_time = time.time() - t0

t1 = time.time(); frozen_pred = frozen_clf.predict(E_te)
infer = 1000*(time.time()-t1)/len(X_te)
evaluate('Frozen DistilBERT + LR', y_te, frozen_pred,
         embed_time + fit_time, E_tr.shape[1]*N_CLASSES, infer)
print(f"(embedding pass took {embed_time:.1f}s of that)")

---
## 6. Fine-tuned DistilBERT

A manual PyTorch training loop rather than the HuggingFace `Trainer`, deliberately:
the `Trainer` API has changed argument names across releases and would silently break
this notebook on a future Colab image. A plain loop is version-stable and every
training decision stays visible — which matters when defending the method in a viva.

Best epoch is selected on **validation macro F1**, and those weights are restored
before the single test-set evaluation.

In [ ]:
class TextDS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length',
                             max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def train_distilbert(Xtr, ytr, Xval, yval, epochs=6, bs=16, lr=2e-5,
                     seed=SEED, verbose=True):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    tr_dl = DataLoader(TextDS(Xtr, ytr), batch_size=bs, shuffle=True)
    va_dl = DataLoader(TextDS(Xval, yval), batch_size=64)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=epochs*len(tr_dl), pct_start=0.1)

    best_f1, best_state, history = -1.0, None, []
    for ep in range(epochs):
        model.train()
        tot = 0.0
        for batch in tr_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            opt.zero_grad()
            out = model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tot += out.loss.item()

        model.eval(); preds, trues = [], []
        with torch.no_grad():
            for batch in va_dl:
                labels = batch.pop('labels')
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                preds.append(model(**batch).logits.argmax(-1).cpu().numpy())
                trues.append(labels.numpy())
        vf1 = f1_score(np.concatenate(trues), np.concatenate(preds), average='macro')
        history.append({'epoch': ep+1, 'train_loss': tot/len(tr_dl), 'val_macro_f1': vf1})
        if verbose:
            print(f"  epoch {ep+1}/{epochs}  loss {tot/len(tr_dl):.4f}  "
                  f"val macro F1 {vf1:.4f}")
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, history, best_f1


@torch.no_grad()
def predict(model, texts, bs=64):
    model.eval()
    dl = DataLoader(TextDS(texts, np.zeros(len(texts), dtype=int)), batch_size=bs)
    out = []
    for batch in dl:
        batch.pop('labels')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out.append(model(**batch).logits.argmax(-1).cpu().numpy())
    return np.concatenate(out)


print("Fine-tuning DistilBERT...")
t0 = time.time()
bert_model, bert_hist, bert_val_f1 = train_distilbert(X_tr, y_tr, X_val, y_val)
bert_time = time.time() - t0

t1 = time.time(); bert_pred = predict(bert_model, X_te)
bert_infer = 1000*(time.time()-t1)/len(X_te)
n_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
evaluate('Fine-tuned DistilBERT', y_te, bert_pred, bert_time, n_params, bert_infer)

---
## 7. Comparison

In [ ]:
comp = pd.DataFrame(RESULTS).T[
    ['accuracy','macro_f1','weighted_f1','trainable_params',
     'train_time_s','inference_ms_per_sample']]
comp = comp.sort_values('macro_f1', ascending=False)
display(comp)

best = comp.index[0]
tfidf_f1 = RESULTS['TF-IDF + LogReg']['macro_f1']
bert_f1 = RESULTS['Fine-tuned DistilBERT']['macro_f1']
gap = bert_f1 - tfidf_f1
cost = (RESULTS['Fine-tuned DistilBERT']['train_time_s']
        / max(RESULTS['TF-IDF + LogReg']['train_time_s'], 0.01))

print(f"\nBest by macro F1: {best}")
print(f"\nDistilBERT minus TF-IDF+LogReg: {gap:+.4f} macro F1")
print(f"Cost multiplier: {cost:.0f}x the training time, "
      f"{RESULTS['Fine-tuned DistilBERT']['trainable_params']:,} trainable parameters")
if gap < 0.02:
    print("\n=> The transformer did NOT meaningfully beat the linear baseline.")
    print("   On 1,200 rows of lexically distinctive text, TF-IDF is sufficient.")
else:
    print("\n=> The transformer earned its cost on this dataset.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
h = pd.DataFrame(bert_hist)
axes[0].plot(h.epoch, h.train_loss, '-o', color='steelblue')
axes[0].set_title('DistilBERT training loss'); axes[0].set_xlabel('epoch')
axes[1].plot(h.epoch, h.val_macro_f1, '-o', color='seagreen', label='DistilBERT (val)')
axes[1].axhline(tfidf_f1, ls='--', color='crimson', label='TF-IDF + LogReg (test)')
axes[1].set_title('Validation macro F1 vs the linear baseline')
axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig('/content/fig_nlp_training.png', bbox_inches='tight'); plt.show()

---
## 8. Where the models actually disagree

Aggregate scores hide the interesting part. This section finds the test cases the
transformer got right and the linear model got wrong, and vice versa. Quote two or
three of these in the report — concrete examples are far more persuasive than a
table of F1 scores.

In [ ]:
bert_ok = bert_pred == y_te
lr_ok = lr_pred == y_te

only_bert = np.where(bert_ok & ~lr_ok)[0]
only_lr = np.where(lr_ok & ~bert_ok)[0]
both_wrong = np.where(~bert_ok & ~lr_ok)[0]

print(f"DistilBERT right, TF-IDF wrong : {len(only_bert)}")
print(f"TF-IDF right, DistilBERT wrong : {len(only_lr)}")
print(f"both wrong                     : {len(both_wrong)}")

def show(idxs, title, k=3):
    print(f"\n{'='*70}\n{title}\n{'='*70}")
    for i in idxs[:k]:
        print(f"\nTEXT : {X_te[i][:200]}")
        print(f"TRUE : {le.classes_[y_te[i]]}")
        print(f"BERT : {le.classes_[bert_pred[i]]}   |   TFIDF: {le.classes_[lr_pred[i]]}")

if len(only_bert): show(only_bert, "DistilBERT correct, TF-IDF wrong")
if len(only_lr):   show(only_lr,   "TF-IDF correct, DistilBERT wrong")
if len(both_wrong): show(both_wrong, "Both wrong (hardest cases)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8.5))
for ax, (name, p) in zip(axes, [('TF-IDF + LogReg', lr_pred),
                                ('Fine-tuned DistilBERT', bert_pred)]):
    cm = confusion_matrix(y_te, p, normalize='true')
    sns.heatmap(cm, cmap='Blues', vmin=0, vmax=1, ax=ax, cbar=False,
                xticklabels=le.classes_, yticklabels=le.classes_,
                annot=False, linewidths=0.3)
    ax.set_title(f"{name} — macro F1 {RESULTS[name]['macro_f1']:.4f}")
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig('/content/fig_nlp_confusion.png', bbox_inches='tight'); plt.show()

---
## 9. Learning curve — when does the transformer start to pay off?

This is the section that turns the phase from a demonstration into a finding.

Both models are retrained on 10%, 25%, 50% and 100% of the training set, across
three random seeds each, and evaluated on the same fixed test set. The crossover
point — if there is one — is where pretrained knowledge begins to outweigh a
sufficient count of labelled examples.

**Runtime: roughly 15–25 minutes.** Leave it running.

In [ ]:
FRACTIONS = [0.10, 0.25, 0.50, 1.00]
SEEDS = [0, 1, 2]
curve = []

for frac in FRACTIONS:
    for sd in SEEDS:
        n = max(int(len(X_tr) * frac), N_CLASSES * 2)
        if frac < 1.0:
            Xs, _, ys, _ = train_test_split(
                X_tr, y_tr, train_size=n, stratify=y_tr, random_state=sd)
        else:
            Xs, ys = X_tr, y_tr

        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(sublinear_tf=True, ngram_range=(1,2),
                                      min_df=1, stop_words='english')),
            ('clf', LogisticRegression(max_iter=2000, C=10, random_state=sd)),
        ]).fit(Xs, ys)
        curve.append({'model': 'TF-IDF + LogReg', 'frac': frac, 'seed': sd,
                      'n_train': len(Xs),
                      'macro_f1': f1_score(y_te, pipe.predict(X_te), average='macro')})

        ep = 6 if frac >= 0.5 else 8   # fewer samples need more passes
        m, _, _ = train_distilbert(Xs, ys, X_val, y_val, epochs=ep,
                                   seed=sd, verbose=False)
        f1b = f1_score(y_te, predict(m, X_te), average='macro')
        curve.append({'model': 'Fine-tuned DistilBERT', 'frac': frac, 'seed': sd,
                      'n_train': len(Xs), 'macro_f1': f1b})
        del m; torch.cuda.empty_cache()

        print(f"  frac {frac:.2f} (n={len(Xs):4d}) seed {sd}: "
              f"TF-IDF {curve[-2]['macro_f1']:.4f}   BERT {f1b:.4f}")

curve_df = pd.DataFrame(curve)
print("\nDone.")

In [ ]:
agg = (curve_df.groupby(['model','n_train'])
       .macro_f1.agg(['mean','std']).reset_index())
display(agg.round(4))

fig, ax = plt.subplots(figsize=(9, 5.8))
for name, colour in [('TF-IDF + LogReg', 'crimson'),
                     ('Fine-tuned DistilBERT', 'steelblue')]:
    s = agg[agg.model == name]
    ax.plot(s.n_train, s['mean'], '-o', lw=2.2, color=colour, label=name)
    ax.fill_between(s.n_train, s['mean']-s['std'], s['mean']+s['std'],
                    color=colour, alpha=0.18)
ax.set_xlabel('training examples'); ax.set_ylabel('test macro F1')
ax.set_title('Learning curve: when does pretraining pay for itself?\n'
             '(mean +/- 1 SD over 3 seeds, fixed test set)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/content/fig_nlp_learning_curve.png',
                                bbox_inches='tight'); plt.show()

piv = agg.pivot(index='n_train', columns='model', values='mean')
piv['BERT_advantage'] = piv['Fine-tuned DistilBERT'] - piv['TF-IDF + LogReg']
print("\nDistilBERT advantage by training size:")
print(piv[['BERT_advantage']].round(4).to_string())

---
## 10. Save everything

In [ ]:
out = {
    'phase': 'DistilBERT vs classical NLP baselines',
    'dataset': 'niyarrbarman/symptom2disease',
    'n_total': int(len(df)), 'n_train': int(len(X_tr)),
    'n_val': int(len(X_val)), 'n_test': int(len(X_te)),
    'n_classes': int(N_CLASSES), 'max_len': MAX_LEN, 'seed': SEED,
    'headline_metric': 'macro_f1',
    'results': RESULTS,
    'distilbert_history': bert_hist,
    'learning_curve': curve_df.to_dict('records'),
}
with open('/content/nlp_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=float)
curve_df.to_csv('/content/nlp_learning_curve.csv', index=False)

bert_model.save_pretrained('/content/distilbert_symptom2disease')
tokenizer.save_pretrained('/content/distilbert_symptom2disease')

print(json.dumps(RESULTS, indent=2))

!zip -q -r /content/nlp_phase_outputs.zip /content/nlp_results.json /content/nlp_learning_curve.csv /content/fig_nlp_*.png
from google.colab import files as gfiles
gfiles.download('/content/nlp_phase_outputs.zip')

---
## 11. Report and viva notes

### The likely result and how to frame it
Expect TF-IDF + Logistic Regression to land close to — possibly level with —
fine-tuned DistilBERT. The 24 conditions are described with distinctive vocabulary
("scaly patches", "burning urination"), so lexical features alone carry most of the
signal. **That is a finding, not a failure.**

Reported honestly, it completes a three-part argument no undergraduate report
usually manages:

| Phase | Data regime | Winner | Why |
|---|---|---|---|
| Tabular | 918 rows, structured | Gradient boosting / TabPFN | Small n, no spatial or sequential structure |
| ECG | 78,798 beats, 1-D signal | CNN | Local morphology; recurrence had nothing to exploit |
| Text | 1,200 docs, 24 classes | *this notebook decides* | Lexical distinctiveness vs semantic depth |

> **"Across three data modalities, the most complex available model won exactly once.
> Architecture selection should follow from the structure and volume of the data,
> not from novelty."**

That covers supervisor points (2), (3), (4), (9) and IPR marker points (3) and (4)
in one paragraph.

### The frozen-vs-fine-tuned control
Report all three DistilBERT conditions. If frozen embeddings underperform TF-IDF
but fine-tuning overtakes it, the gain came from **task adaptation**, not from
pretrained knowledge — a precise claim most reports cannot make because they never
ran the control.

### Limitations to state before a marker finds them
1. **1,200 rows, 50 per class.** Small. Confidence intervals on macro F1 are wide;
   the learning curve's seed spread quantifies this — use it rather than reporting
   a single number as if it were exact.
2. **The text is not real clinical data.** These are clean, single-condition,
   first-person descriptions. Real intake text has comorbidities, negation
   ("no chest pain"), misspellings and ambiguity. Do not claim clinical readiness.
3. **24 conditions is a closed set.** The model cannot say "I don't know", and will
   confidently assign one of 24 labels to any input, including a condition it has
   never seen. For a triage tool that is a genuine safety concern and belongs in
   your ethics section, tied to a concrete mitigation (a confidence threshold below
   which the system defers to a human).
4. **No external validation.** Unlike the cardiovascular phase, there is no second
   symptom dataset here, so the generalisation question stays open. Say so
   explicitly and reference the cross-dataset chapter, where you *did* measure it.

### Integration into HealthGuard AI
Keep this as a separate `/api/predict/symptoms` endpoint returning a ranked
shortlist with confidence scores — never a single diagnosis. That directly implements
IPR marker point (2), which asked how the system stays *supportive rather than
diagnostic*.